# A00 - 进阶 PINN 通用代码块

后续流体传热和流固耦合案例都会反复用到这些组件：

- 多输入多输出 MLP
- 一阶、二阶导数计算
- 矩形区域采样
- 二维场可视化

你可以把本 notebook 当作代码积木库。

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(100)
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_dim, out_dim, width=64, depth=5):
        super().__init__()
        layers = [nn.Linear(in_dim, width), nn.Tanh()]
        for _ in range(depth - 1):
            layers += [nn.Linear(width, width), nn.Tanh()]
        layers += [nn.Linear(width, out_dim)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

def grad(y, x):
    return torch.autograd.grad(y, x, torch.ones_like(y), create_graph=True)[0]

def sample_rectangle(n, x_min=0.0, x_max=1.0, y_min=0.0, y_max=1.0):
    x = x_min + (x_max - x_min) * torch.rand(n, 1, device=device)
    y = y_min + (y_max - y_min) * torch.rand(n, 1, device=device)
    return x, y

def make_grid(nx=80, ny=80, x_min=0.0, x_max=1.0, y_min=0.0, y_max=1.0):
    xs = torch.linspace(x_min, x_max, nx, device=device)
    ys = torch.linspace(y_min, y_max, ny, device=device)
    X, Y = torch.meshgrid(xs, ys, indexing="ij")
    return X, Y, torch.cat([X.reshape(-1, 1), Y.reshape(-1, 1)], dim=1)

def plot_field(field, title, extent=(0, 1, 0, 1)):
    plt.imshow(field.detach().cpu().T, origin="lower", extent=extent, aspect="auto")
    plt.colorbar()
    plt.title(title)

## 使用建议

复杂多物理 PINN 不要一开始就把所有项都打开。推荐顺序：

1. 先只训练边界条件，确认输出量级合理。
2. 再加入 PDE 残差。
3. 最后加入界面、数据、反演参数等约束。
4. 每次画出每个 loss term，不只看 total loss。